# 00 — Setup
Her Colab session'ının başında **çalıştır**.
1. Drive mount
2. GitHub'dan repo pull
3. requirements.txt install
4. Drive dizinleri oluştur
5. .env dosyası yaz
6. GPU bilgisi

In [ ]:
# 1) Google Drive mount
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2) Repo klonla veya güncelle
import os
REPO_URL = 'https://github.com/hakanskn/CALMA.git'
REPO_DIR = '/content/arf_prototype'

if not os.path.exists(os.path.join(REPO_DIR, 'src')):
    ret = os.system(f'git clone {REPO_URL} /content/CALMA')
    if ret != 0:
        raise RuntimeError(
            "git clone başarısız oldu!\n"
            "Repo muhtemelen private. Seçenekler:\n"
            "  1. Repo'yu public yapıp tekrar çalıştır\n"
            "  2. URL'yi token'lı kullan: https://<TOKEN>@github.com/hakanskn/CALMA.git\n"
            "  3. Sol panelden 'Files > Upload' ile dosyaları /content/arf_prototype/ altına yükle"
        )
    os.system('cp -r /content/CALMA/arf_prototype /content/arf_prototype')
else:
    ret = os.system('cd /content/CALMA && git pull && cp -r arf_prototype/* /content/arf_prototype/')
    if ret != 0:
        print('git pull başarısız, mevcut dosyalarla devam ediliyor.')

print(f'REPO_DIR içeriği: {os.listdir(REPO_DIR)}')

REPO_DIR içeriği: ['README.md', 'notebooks', 'configs', 'batch_run.py', '.env.example', 'scripts', 'src', 'train.py', 'dashboard', 'requirements.txt', 'results', '.gitignore']


In [ ]:
# 3) Requirements
import os, pathlib, sys

try:
    import google.colab  # noqa
    REPO_DIR = '/content/arf_prototype'
except ImportError:
    # Walk up from CWD until we find requirements.txt
    _cwd = pathlib.Path.cwd()
    for _p in [_cwd] + list(_cwd.parents):
        if (_p / 'requirements.txt').exists():
            REPO_DIR = str(_p)
            break
        if (_p / 'arf_prototype' / 'requirements.txt').exists():
            REPO_DIR = str(_p / 'arf_prototype')
            break
    else:
        raise FileNotFoundError(f"requirements.txt not found from {_cwd}")

_req = os.path.join(REPO_DIR, 'requirements.txt')
print(f'REPO_DIR: {REPO_DIR}')
print(f'Installing from: {_req}')
if sys.platform == 'win32':
    os.system(f'pip install -q -r "{_req}"')
else:
    os.system(f'pip install -q -r {_req}')

REPO_DIR: /content/arf_prototype
Installing from: /content/arf_prototype/requirements.txt


In [ ]:
# 4) Drive dizinleri
DRIVE_BASE = '/content/drive/MyDrive/arf_results'
for sub in ['runs', 'comparisons']:
    os.makedirs(f'{DRIVE_BASE}/{sub}', exist_ok=True)
print('Drive base:', DRIVE_BASE)

Drive base: /content/drive/MyDrive/arf_results


In [ ]:
# 5) .env
os.makedirs(REPO_DIR, exist_ok=True)
env = f'DRIVE_BASE={DRIVE_BASE}\nHF_HOME={DRIVE_BASE}/hf_cache\n'
with open(f'{REPO_DIR}/.env', 'w') as f:
    f.write(env)
os.environ['DRIVE_BASE'] = DRIVE_BASE
os.environ['HF_HOME'] = f'{DRIVE_BASE}/hf_cache'
print('.env yazıldı.')

.env yazıldı.


In [ ]:
# 6) GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# 7) Sanity check — paketleri import et
import sys
sys.path.insert(0, REPO_DIR)
from src.config import ExperimentConfig, list_methods
print('Available methods:', list_methods())
print('Default config:', ExperimentConfig.load('baseline').method)

Available methods: ['baseline', 'A1_PCR', 'A2_LI_SCR', 'A3_RBR', 'B1_MI_S3T', 'B2_CMA', 'B3_FWML', 'C1_PCUN', 'C2_EP_T', 'C3_CHA']
Default config: baseline
